# 🧪 TÁI LẬP VÀ ĐÁNH GIÁ MÔ HÌNH BASELINE (CVPRW 2019)
### Đề tài: Early Prediction of Driver Drowsiness using Computer Vision
**Bài báo gốc:** *A Realistic Dataset and Baseline Temporal Model for Early Drowsiness Detection* (Reza Ghoddoosian et al., CVPRW 2019)

---
**Quy trình tự động:** Bấm menu **Runtime** -> **Run all** (hoặc `Ctrl + F9`), Colab sẽ tự chạy toàn bộ từ Bước 1 đến Bước 7.

## Bước 1: Thiết lập môi trường độc lập (Python 3.7 + TensorFlow 1.15 + Protobuf 3.20)

In [ ]:
%%bash
# 1. Tải và cài đặt Miniconda nếu chưa có
if [ ! -f /root/miniconda/bin/python ]; then
    wget -q -O miniconda.sh https://repo.anaconda.com/miniconda/Miniconda3-py37_4.12.0-Linux-x86_64.sh
    bash miniconda.sh -b -f -p /root/miniconda
fi

# 2. Cài đặt các gói chuẩn xác cho TensorFlow 1.15
/root/miniconda/bin/pip install -q tensorflow==1.15.0 "numpy<1.19.0" "protobuf<=3.20.3" scipy==1.1.0 scikit-learn==0.20.0 matplotlib pandas

# 3. Kiểm tra xác nhận
/root/miniconda/bin/python -c "import tensorflow as tf; import google.protobuf; print('Moi truong san sang -> TF:', tf.__version__, '| Protobuf:', google.protobuf.__version__)"

## Bước 2: Tải mã nguồn gốc và dữ liệu Benchmark của tác giả

In [ ]:
%%bash
mkdir -p /content/Early-Drowsiness-Detection
if [ ! -f /content/Early-Drowsiness-Detection/HMSLSTM_Main.py ]; then
    rm -rf /content/Early-Drowsiness-Detection
    git clone https://github.com/rezaghoddoosian/Early-Drowsiness-Detection.git /content/Early-Drowsiness-Detection
fi
echo "Da san sang ma nguon tai /content/Early-Drowsiness-Detection"

## Bước 3: Tạo mã đánh giá độc lập 5-Fold trên Checkpoint gốc

In [ ]:
%%writefile /content/Early-Drowsiness-Detection/evaluate_baseline.py
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
import sys
sys.path.insert(0, '/content/Early-Drowsiness-Detection')
import numpy as np
import tensorflow as tf
import pandas as pd
import HMSLSTM_Main as Main

print("TensorFlow Version in evaluation:", tf.__version__)

def batchNorm(x, beta, gamma, training, scope='bn'):
    with tf.variable_scope(scope):
        batch_mean, batch_var = tf.nn.moments(x, [0], name='moments')
        ema = tf.train.ExponentialMovingAverage(decay=0.5)
        def mean_var_with_update():
            ema_apply_op = ema.apply([batch_mean, batch_var])
            with tf.control_dependencies([ema_apply_op]):
                return tf.identity(batch_mean), tf.identity(batch_var)
        mean, var = tf.cond(training, mean_var_with_update, lambda: (ema.average(batch_mean), ema.average(batch_var)))
        normed = tf.nn.batch_normalization(x, mean, var, beta, gamma, 1e-3)
    return normed

def Network(input, Pre_fc1_size, Post_fc1_size_per_layer, embb_size, embb_size2, Post_fc2_size, hstate_size, num_layers, feature_size, step_size, output_size, keep_p, training):
    end_points = {}
    batch_size = tf.shape(input)[0]
    with tf.variable_scope('pre_fc1'):
        pre_fc1_weights = tf.get_variable('weights', [feature_size, Pre_fc1_size], dtype=tf.float32,
                                          initializer=tf.contrib.layers.xavier_initializer(uniform=False, seed=None, dtype=tf.float32))
        pre_fc1_biases = tf.get_variable('biases', [Pre_fc1_size], dtype=tf.float32,
                                         initializer=tf.constant_initializer(0.0))
        reshaped_input_net = tf.reshape(input, [-1, feature_size])
        input_RNN = tf.matmul(reshaped_input_net, pre_fc1_weights)
        input_RNN = batchNorm(input_RNN, pre_fc1_biases, None, training, scope='bn')
        input_RNN = tf.nn.relu(input_RNN)
        input_RNN = tf.reshape(input_RNN, [-1, step_size, Pre_fc1_size])
        input_RNN = tf.nn.dropout(input_RNN, keep_p)

    end_points['pre_fc1'] = input_RNN
    hmslstm_block = Main.HMSLSTM_Block(input_size=[batch_size, step_size, Pre_fc1_size], step_size=step_size,
                                      hstate_size=hstate_size, num_layers=num_layers, keep_p=keep_p)
    output_RNN_set, states_RNN, concati = hmslstm_block(input_RNN, reuse=False)
    end_points['mid_layers'] = output_RNN_set

    with tf.variable_scope('post_fc1'):
        for lay in range(num_layers):
            post_fc1_weights = tf.get_variable('weights_%s' % lay, [hstate_size[lay], Post_fc1_size_per_layer], dtype=tf.float32,
                                               initializer=tf.contrib.layers.xavier_initializer(uniform=False, seed=None, dtype=tf.float32))
            post_fc1_biases = tf.get_variable('biases_%s' % lay, [Post_fc1_size_per_layer], dtype=tf.float32,
                                              initializer=tf.constant_initializer(0.0))
            trash, output_RNN = tf.split(output_RNN_set[lay], [step_size - 1, 1], axis=0, name='layers')
            output_RNN = tf.squeeze(output_RNN, axis=0)
            post_fc1 = tf.matmul(output_RNN, post_fc1_weights)
            post_fc1 = batchNorm(post_fc1, post_fc1_biases, None, training, scope='bn')
            if lay == 0:
                post_fc1_out = post_fc1
            else:
                post_fc1_out = tf.concat([post_fc1_out, post_fc1], axis=1)

        post_fc1_out = tf.nn.relu(post_fc1_out)
        post_fc1_out = tf.nn.dropout(post_fc1_out, keep_p)
        end_points['post_fc1'] = post_fc1_out

    with tf.variable_scope('embeddings'):
        embeddings_weights = tf.get_variable('weights', [Post_fc1_size_per_layer * num_layers, embb_size], dtype=tf.float32,
                                             initializer=tf.contrib.layers.xavier_initializer(uniform=False, seed=None, dtype=tf.float32))
        embeddings_biases = tf.get_variable('biases', [embb_size], dtype=tf.float32,
                                           initializer=tf.constant_initializer(0.0))
        embeddings = tf.matmul(post_fc1_out, embeddings_weights)
        embeddings = batchNorm(embeddings, embeddings_biases, None, training, scope='bn')
        embeddings = tf.nn.relu(embeddings)
        embeddings = tf.nn.dropout(embeddings, keep_p)
        end_points['embeddings'] = embeddings

    with tf.variable_scope('embeddings2'):
        embeddings_weights2 = tf.get_variable('weights', [embb_size, embb_size2], dtype=tf.float32,
                                              initializer=tf.contrib.layers.xavier_initializer(uniform=False, seed=None, dtype=tf.float32))
        embeddings_biases2 = tf.get_variable('biases', [embb_size2], dtype=tf.float32,
                                            initializer=tf.constant_initializer(0.0))
        embeddings2 = tf.matmul(embeddings, embeddings_weights2)
        embeddings2 = batchNorm(embeddings2, embeddings_biases2, None, training, scope='bn')
        embeddings2 = tf.nn.relu(embeddings2)
        embeddings2 = tf.nn.dropout(embeddings2, keep_p)
        end_points['embeddings2'] = embeddings2

    with tf.variable_scope('post_fc2'):
        post_fc2_weights = tf.get_variable('weights', [embb_size2, Post_fc2_size], dtype=tf.float32,
                                           initializer=tf.contrib.layers.xavier_initializer(uniform=False, seed=None, dtype=tf.float32))
        post_fc2_biases = tf.get_variable('biases', [Post_fc2_size], dtype=tf.float32,
                                          initializer=tf.constant_initializer(0.0))
        post_fc2_out = tf.matmul(embeddings2, post_fc2_weights)
        post_fc2_out = batchNorm(post_fc2_out, post_fc2_biases, None, training, scope='bn')
        post_fc2_out = tf.nn.relu(post_fc2_out)
        post_fc2_out = tf.nn.dropout(post_fc2_out, keep_p)
        end_points['post_fc2'] = post_fc2_out

    with tf.variable_scope('last_fc'):
        last_fc_weights = tf.get_variable('weights', [Post_fc2_size, output_size], dtype=tf.float32,
                                          initializer=tf.contrib.layers.xavier_initializer(uniform=False, seed=None, dtype=tf.float32))
        last_fc_biases = tf.get_variable('biases', [output_size], dtype=tf.float32,
                                         initializer=tf.constant_initializer(0.0))
        output = tf.matmul(post_fc2_out, last_fc_weights) + last_fc_biases
        if output_size == 1:
            end_points['before the last sigmoid'] = output
            output = 10 * tf.sigmoid(output)

    return output, end_points, concati

all_records = []
labels_pool = np.array([0, 5, 10])

for fold_idx in range(5):
    fold_num = fold_idx + 1
    print("\n" + "="*60)
    print("--- DANG DANH GIA TEST FOLD %d VOI CHECKPOINT my_model%d ---" % (fold_num, fold_num))
    print("="*60)
    
    Blinks = np.load('/content/Early-Drowsiness-Detection/Blinks_30_Fold%d.npy' % fold_num)
    Labels = np.load('/content/Early-Drowsiness-Detection/Labels_30_Fold%d.npy' % fold_num)
    BlinksTest = np.load('/content/Early-Drowsiness-Detection/BlinksTest_30_Fold%d.npy' % fold_num)
    LabelsTest = np.load('/content/Early-Drowsiness-Detection/LabelsTest_30_Fold%d.npy' % fold_num)
    
    for feat in range(4):
        m = np.mean(Blinks[:, :, feat])
        s = np.std(Blinks[:, :, feat])
        BlinksTest[:, :, feat] = (BlinksTest[:, :, feat] - m) / s
        Blinks[:, :, feat] = (Blinks[:, :, feat] - m) / s
        
    tf.reset_default_graph()
    input_net = tf.placeholder(tf.float32, shape=(None, None, 4), name='bacth_in')
    labels = tf.placeholder(tf.float32, shape=(None, 1), name='labels_net')
    keep_p = tf.placeholder(tf.float32)
    training = tf.placeholder(tf.bool, name='phase_train')
    
    output, end_points, concati = Network(
        input=input_net, Pre_fc1_size=32, Post_fc1_size_per_layer=16,
        embb_size=16, embb_size2=16, Post_fc2_size=8, hstate_size=[32,32,32,32],
        num_layers=4, feature_size=4, step_size=30, output_size=1, keep_p=keep_p, training=training
    )
    
    saver = tf.train.Saver()
    with tf.Session() as sess:
        saver.restore(sess, '/content/Early-Drowsiness-Detection/my_model%d' % fold_num)
        predicts_raw = sess.run(output, feed_dict={input_net: BlinksTest, labels: LabelsTest, keep_p: 1.0, training: False})
        
        np.clip(predicts_raw, 0, 10, out=predicts_raw)
        predicted_indices = (predicts_raw // 3.34).astype(np.int8)
        predicted_labels = labels_pool[predicted_indices].flatten()
        true_labels = LabelsTest.flatten()
        raw_scores = predicts_raw.flatten()
        
        correct_count = np.sum(predicted_labels == true_labels)
        total_count = len(true_labels)
        print("-> Do chinh xac Fold %d (BSA): %d/%d (%.2f%%)" % (fold_num, correct_count, total_count, correct_count/total_count*100.0))
        
        for idx in range(total_count):
            mean_feat = BlinksTest[idx].mean(axis=0)
            all_records.append({
                'Fold': fold_num,
                'Sample_Index': idx,
                'Ground_Truth': true_labels[idx],
                'Predicted_Label': predicted_labels[idx],
                'Predicted_Score': round(float(raw_scores[idx]), 2),
                'Is_Correct': bool(predicted_labels[idx] == true_labels[idx]),
                'Mean_Freq': round(float(mean_feat[0]), 3),
                'Mean_Amp': round(float(mean_feat[1]), 3),
                'Mean_Dur': round(float(mean_feat[2]), 3),
                'Mean_Vel': round(float(mean_feat[3]), 3)
            })

df = pd.DataFrame(all_records)
df.to_csv('/content/Early-Drowsiness-Detection/all_evaluation_results.csv', index=False)

wrong_df = df[df['Is_Correct'] == False]
wrong_df.to_csv('/content/Early-Drowsiness-Detection/wrong_testcases.csv', index=False)

print("\n" + "="*60)
print("HOAN THANH: Da kiem thu tong cong %d chuoi tren ca 5 Fold." % len(df))
correct_len = len(df[df['Is_Correct'] == True])
wrong_len = len(wrong_df)
print("So mau doan DUNG: %d (%.2f%%)" % (correct_len, correct_len/len(df)*100.0))
print("So mau doan SAI (Failure Cases): %d (%.2f%%)" % (wrong_len, wrong_len/len(df)*100.0))
print("Da xuat file: /content/Early-Drowsiness-Detection/wrong_testcases.csv")
print("="*60)


## Bước 4: Thực thi đánh giá (Chạy bằng môi trường TensorFlow 1.15)

In [ ]:
!cd /content/Early-Drowsiness-Detection && PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python /root/miniconda/bin/python evaluate_baseline.py

## Bước 5: Vẽ Ma trận nhầm lẫn (Confusion Matrix Heatmap)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('/content/Early-Drowsiness-Detection/all_evaluation_results.csv')

classes = [0.0, 5.0, 10.0]
class_names = ['Tinh tao (0)', 'Buon ngu nhe (5)', 'Buon ngu sau (10)']
cm = np.zeros((3, 3), dtype=int)

for i, true_c in enumerate(classes):
    for j, pred_c in enumerate(classes):
        cm[i, j] = np.sum((df['Ground_Truth'] == true_c) & (df['Predicted_Label'] == pred_c))

fig, ax = plt.subplots(figsize=(8, 6))
cax = ax.matshow(cm, cmap='Blues', alpha=0.85)

for i in range(3):
    for j in range(3):
        total_in_class = np.sum(cm[i, :])
        pct = cm[i, j] / total_in_class * 100
        color = 'white' if cm[i, j] > np.max(cm)/2 else 'black'
        if i != j and pct > 15:
            color = 'red'
        ax.text(j, i, f"{cm[i, j]}\n({pct:.1f}%)", ha='center', va='center', 
                fontsize=12, fontweight='bold', color=color)

fig.colorbar(cax)
ax.set_xticks([0, 1, 2])
ax.set_yticks([0, 1, 2])
ax.set_xticklabels(class_names, fontsize=11, fontweight='bold')
ax.set_yticklabels(class_names, fontsize=11, fontweight='bold')
plt.xlabel('Nhan Mo hinh Du doan (Predicted)', fontsize=13, fontweight='bold')
plt.ylabel('Nhan Thuc te cua Tai xe (Ground Truth)', fontsize=13, fontweight='bold')
plt.title('MA TRAN NHAM LAN MO HINH BASELINE (CVPRW 2019)\n', fontsize=14, fontweight='bold')
plt.savefig('/content/Early-Drowsiness-Detection/confusion_matrix_baseline.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("--- THONG KE TY LE DU DOAN THEO TUNG MUC DO BUON NGU ---")
for i, name in enumerate(class_names):
    total = np.sum(cm[i, :])
    correct = cm[i, i]
    wrong = total - correct
    print(f"{name}: Tong {total} mau | Dung {correct} ({correct/total*100:.2f}%) | SAI {wrong} ({wrong/total*100:.2f}%)")
print("="*60)

## Bước 6: Phân tích các ca đoán sai nguy hiểm (Bỏ sót buồn ngủ)

In [ ]:
wrong_df = pd.read_csv('/content/Early-Drowsiness-Detection/wrong_testcases.csv')
dangerous_misses = wrong_df[(wrong_df['Ground_Truth'] == 5.0) & (wrong_df['Predicted_Label'] == 0.0)]
print(f"So ca BO SOT NGUY HIEM (Thuc te Buon ngu nhe nhung mo hinh bao Tinh tao): {len(dangerous_misses)} ca!")
dangerous_misses[['Fold', 'Sample_Index', 'Ground_Truth', 'Predicted_Score', 'Predicted_Label', 'Mean_Freq', 'Mean_Dur']].head(15)

## Bước 7: Tải các file kết quả về máy tính cá nhân

In [ ]:
from google.colab import files
try:
    files.download('/content/Early-Drowsiness-Detection/wrong_testcases.csv')
    files.download('/content/Early-Drowsiness-Detection/confusion_matrix_baseline.png')
    files.download('/content/Early-Drowsiness-Detection/all_evaluation_results.csv')
    print("Dang tai 3 file ve may...")
except Exception as e:
    print("Luu y: Neu trinh duyet chan tai nhieu file, ban co the tai truc tiep tu muc Files ben trai cua Colab:", e)